# Pré-processamento dos dados

---

## 1. Carregando e separando os dados

In [2]:
RANDOM_SEED = 42

In [3]:
import pandas as pd

leads = pd.read_csv('data/churn_aplicacao_leads_final.csv')
pedidos = pd.read_csv('data/churn_comportamental_pedidos_final.csv')

total = leads.merge(right=pedidos, on='id_cliente', how='left')

Vamos treinar dois modelos: um **comportamental** e um de **análise de leads**.

No modelo comportamental, o conjunto de treinamento será referente aos clientes que tem pedidos correspondentes na tabela de pedidos, enquanto que o conjunto de treinamento do modelo de análise de leads conterá apenas dados de clientes que nunca fizeram um pedido antes. Nesse último caso, nenhum atributo da tabela de pedidos será relevante, então podemos também removê-los dos atributos disponíveis.

Com base nisso, separamos os conjuntos usando essas regras:

In [4]:
leads_raw: pd.DataFrame = total[total.id_pedido.isna()]           # Clientes que nunca pediram
comportamental_raw: pd.DataFrame = total[total.id_pedido.notna()] # Clientes que tem pedidos

leads_raw = leads_raw.drop(['id_pedido', 'valor', 'produtos', 'data_entrega'], axis='columns') # Removendo atributos referentes a pedidos

Por fim, vamos definir uma porcentagem para a divisão entre treino + validação e teste:

In [5]:
TRAIN_VAL_RATIO = 0.8
TEST_RATIO      = 0.2

### 1.1. Separando o conjunto de dados `leads_raw`

Sem levar os pedidos em consideração, temos um conjunto de dados que não apresenta uma relação de temporalidade relevante, então a sua divisão segue de modo padrão: faremos uma divisão estratificada no atributo-alvo, o `churned`:

In [6]:
from sklearn.model_selection import train_test_split

leads_raw_X, leads_raw_y = leads_raw, leads_raw.pop('churned')

leads_raw_train_val_X, leads_raw_test_X, leads_raw_train_val_y, leads_raw_test_y = train_test_split(
    leads_raw_X,
    leads_raw_y,
    train_size=TRAIN_VAL_RATIO,
    test_size=TEST_RATIO,
    random_state=RANDOM_SEED,
    stratify=leads_raw_y
)

LEADS = pd.DataFrame({ # pondo tudo num DataFrame só pela facilidade de acesso
    'train_val': {
        'X': leads_raw_train_val_X,
        'y': leads_raw_train_val_y,
    },
    'test': {
        'X': leads_raw_test_X,
        'y': leads_raw_test_y,
    },
})

print(f'Tamanho do conjunto de treino + validação: {LEADS.train_val.X.shape[0]} linhas x {LEADS.train_val.X.shape[1]} colunas')
print(f'Tamanho do conjunto de teste:              {LEADS.test.X.shape[0]} linhas x {LEADS.test.X.shape[1]} colunas')

Tamanho do conjunto de treino + validação: 800 linhas x 18 colunas
Tamanho do conjunto de teste:              200 linhas x 18 colunas


### 1.2. Separando o conjunto de dados `comportamental_raw`

Como o conjunto de dados comportamental leva em consideração os pedidos dos clientes, temos uma série temporal, como discutido no processo de EDA.

Para separar os dados agora, é necessário evitar que o futuro seja usado como treinamento para testes no passado, então, devemos ordenar os dados com base na data. A partir disso, basta escolher uma data de corte e separar os dois conjuntos adequadamente.

Queremos manter a mesma proporção de amostras de teste que o conjunto de análise de leads, então usaremos `TEST_RATIO` do intervalo total (90 dias) para definir a janela de corte:

In [7]:
TEST_TIMEFRAME = pd.to_timedelta(90 * TEST_RATIO, 'days')

Agora, basta separar ambos conjuntos:

In [8]:
comportamental_raw.data_entrega = comportamental_raw.data_entrega.apply(pd.to_datetime)

ordered = comportamental_raw.sort_values(by='data_entrega')

date_threshold = pd.to_datetime(ordered.data_entrega.max()) - TEST_TIMEFRAME

comportamental_raw_train_val = ordered[ordered.data_entrega < date_threshold]
comportamental_raw_test = ordered[ordered.data_entrega >= date_threshold]

COMPORTAMENTAL = pd.DataFrame({ # pondo tudo num DataFrame só pela facilidade de acesso
    'train_val': {
        'X': comportamental_raw_train_val,
        'y': comportamental_raw_train_val.pop('churned'),
    },
    'test': {
        'X': comportamental_raw_test,
        'y': comportamental_raw_test.pop('churned'),
    },
})

print(f'Tamanho do conjunto de treino + validação: {COMPORTAMENTAL.train_val.X.shape[0]} linhas x {COMPORTAMENTAL.train_val.X.shape[1]} colunas')
print(f'Tamanho do conjunto de teste:              {COMPORTAMENTAL.test.X.shape[0]}  linhas x {COMPORTAMENTAL.test.X.shape[1]} colunas')

Tamanho do conjunto de treino + validação: 3554 linhas x 22 colunas
Tamanho do conjunto de teste:              887  linhas x 22 colunas


/tmp/ipykernel_253294/3646340780.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  comportamental_raw.data_entrega = comportamental_raw.data_entrega.apply(pd.to_datetime)


Verificando as proporções finais e os intervalos de tempo a serem utilizados:

In [9]:
print('Porcentagem da divisão dos dados comportamentais:')

print(f'{100 * COMPORTAMENTAL.train_val.X.shape[0] / comportamental_raw.shape[0]:.2f}% para treino e validação')
print(f'       |-> Intervalo de tempo: {90 - TEST_TIMEFRAME.days} dias')

print(f'{100 * COMPORTAMENTAL.test.X.shape[0] / comportamental_raw.shape[0]:.2f}% para teste')
print(f'       |-> Intervalo de tempo: {TEST_TIMEFRAME.days} dias')

Porcentagem da divisão dos dados comportamentais:
80.03% para treino e validação
       |-> Intervalo de tempo: 72 dias
19.97% para teste
       |-> Intervalo de tempo: 18 dias


---

# 2. Pré-processamento dos dados

In [10]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline

### 2.1. Conjunto de análise de leads

Aqui vamos definir as etapas de pré-processamento para todos os dados referentes ao cliente isoladamente, sem levar em conta seu comportamento. 

O pipeline que criarmos no final será reutilizado para o conjunto comportamental!

#### 2.1.1. Removendo atributos

Começamos removendo os seguintes atributos:
* `id_cliente`: identificador irrelevante
* `uf`: não existe mais de uma UF no conjunto de dados
* `google_maps_avaliacao`, `google_maps_contagem_avaliacoes`: faltante em quase 99% das amostras, não têm semântica clara para nulo
* `cnae_codigo`: redundante com `segmento_cliente`. Na EDA, vimos que a maioria dos segmentos se concentra em uma única divisão do CNAE, mas o contrário não acontece (a divisão 53, por exemplo, reúne vários segmentos com taxas de churn entre 33% e 55%). Ou seja, o segmento já carrega a informação do CNAE e ainda separa melhor o churn
*Obs.: Os atributos `ifood_contagem_avaliacoes` e `ifood_faixa_preco` serão removidos no próximo passo.*

In [11]:
TO_DROP = ['id_cliente', 'uf', 'google_maps_avaliacao', 'google_maps_contagem_avaliacoes','cnae_codigo']

#### 2.1.2. Extrair features dos valores do iFood/Google Maps

Vamos remover os atributos que carregam informações mais específicas do cliente acerca da sua presença no iFood e/ou Google Maps, mas ainda queremos saber se o cliente está no iFood ou não, e se tem website no Google Maps ou não.

Para isso, vamos corrigir os valores nulos de `google_maps_tem_website` e criar uma nova feature, `tem_ifood`:

In [12]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer


def extract_tem_ifood(X: pd.DataFrame):
    X['tem_ifood'] = (X.ifood_faixa_preco.notna() | X.ifood_contagem_avaliacoes.notna()).astype(int)
    X.drop(columns=['ifood_contagem_avaliacoes', 'ifood_faixa_preco'], inplace=True)
    
    return X

IFOOD_FEATURE_EXTRACTOR = FunctionTransformer(extract_tem_ifood, feature_names_out=lambda _, __: ['tem_ifood'])

# sabemos que google_maps_tem_website ou é 1 ou é nan,
# então só basta imputar onde tem nan => 0
GOOGLE_MAPS_IMPUTER = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=0)

#### 2.1.3. Lidando com faixas de valores

Já que vamos remover os atributos `ifood_contagem_avaliacoes`, `google_maps_avaliacao` e `google_maps_contagem_avaliacoes`, os atributos que ainda representam faixas de valores no conjunto de leads são **`capital_social`** e **`idade_cnpj`**.

Nesses casos, vamos fazer um *label encoding* desses valores para manter a relação de ordem entre intervalos:

In [13]:
from sklearn.preprocessing import OrdinalEncoder


def get_ordered_ranges(data: pd.Series):
    get_lower_bound = lambda str: float(str.strip('()[]').split(',')[0])
    return sorted(data.dropna().unique(), key=get_lower_bound)

CAPITAIS_SOCIAIS = get_ordered_ranges(LEADS.train_val.X.capital_social)
IDADES_CNPJ = get_ordered_ranges(LEADS.train_val.X.idade_cnpj)

CAPITAL_SOCIAL_ENCODER = OrdinalEncoder(categories=[CAPITAIS_SOCIAIS])
IDADE_CNPJ_ENCODER = OrdinalEncoder(categories=[IDADES_CNPJ])

#### 2.1.4. Lidando com valores numéricos com outliers estatísticos

Como visto na EDA, os atributos `serasa_contagem_negativacoes` e `serasa_contagem_protestos` têm valores geralmente perto de 0, mas possuem alguns outliers estatísticos muito acima desse nível.

Não queremos remover esses valores, então, como aprendemos nas aulas de EDA, aplicaremos uma função logaritmo nesses valores:

In [14]:
# vamos usar a função do numpy correspondente a f(x) = ln(1+x)
# pois os valores de x podem ser 0 e queremos valores reais
LOG_TRANSFORMER = FunctionTransformer(np.log1p, feature_names_out='one-to-one')

#### 2.1.5. Agrupando categorias de valores

O atributo `natureza_juridica` tem duas categorias primárias e outras duas que tem uma característica comum com as majoritárias. Podemos agrupar as categorias `230-5` e `213-5`, e as categorias `206-2` e `224-0`, conforme analisamos na EDA:

In [15]:
def transform_natureza_juridica(X: pd.DataFrame):
    X.natureza_juridica = X.natureza_juridica.str[:5].isin(['206-2', '224-0']).astype(int)
    return X

NATUREZA_JURIDICA_TRANSFORMER = FunctionTransformer(transform_natureza_juridica, feature_names_out='one-to-one')

#### 2.1.7. Extraindo informações de `serasa_credores`

Vamos extrair a quantidade de credores na "lista" da coluna:

In [16]:
import re


def extract_qtd_credores(X: pd.DataFrame):
    X.serasa_credores = X.serasa_credores.apply(lambda str: len(re.split(r',\s', str)) if pd.notna(str) else 0)
    return X

QTD_CREDORES_EXTRACTOR = FunctionTransformer(extract_qtd_credores, feature_names_out='one-to-one')

# TODO: definir se vamos fazer um encoding do tipo de credor que o cliente possui
# menos de 10% dos clientes no geral tinham credor.. vale a pena?

# na minha opinião não, além de que nenhuma váriavel se relacionou com churn no EDA, acho que deixaria grupos muitos pequenos

# TODO: decidir se mantemos serasa_socio_tem_negativacao --> vamos manter

#### 2.1.8. Codificando outras variáveis categóricas

- `fonte_cliente`: faremos um one-hot encoding simples, são apenas 5 categorias para essa variável;

In [17]:
from sklearn.preprocessing import OneHotEncoder

FONTES_CLIENTE = [f'Fonte {n}' for n in range(1, 6)]
FONTE_CLIENTE_ENCODER = OneHotEncoder(categories=[FONTES_CLIENTE], feature_name_combiner=lambda feature, val: f'{feature}_{val[-1]}')

- `segmento_cliente`: faremos um one-hot encoding apenas com os 14 segmentos que tinham pelo menos 30 leads na EDA. Os 7 segmentos mais raros irá funcionar como uma categoria "Outros", já que estimar cada um separadamente, com tão poucos leads, faria o modelo aprender ruído;

In [18]:
# segmentos com pelo menos 30 leads,com base na EDA -->14 dos 21
SEGMENTOS = [f'Segmento {n}' for n in [2, 3, 4, 5, 8, 10, 11, 13, 14, 15, 16, 18, 19, 20]]

SEGMENTO_ENCODER = OneHotEncoder(categories=[SEGMENTOS],handle_unknown='ignore',feature_name_combiner=lambda feature, val: f'{feature}_{val.split()[-1]}')

- `municipio`: agrupamos os 31 municípios em 3 regiões (Fortaleza,Região Metropolitana de Fortaleza (RMF) e Interior), seguindo a composição oficial da RMF, e depois aplicamos um one-hot encoding com essas 3 categorias.

In [ ]:
RMF= ['CAUCAIA', 'MARACANAU', 'MARANGUAPE', 'PACATUBA', 'ITAITINGA', 'EUSEBIO', 'AQUIRAZ', 'GUAIUBA', 'HORIZONTE','PACAJUS', 'CHOROZINHO', 'SAO GONCALO DO AMARANTE', 'PINDORETAMA', 'CASCAVEL', 'PARACURU', 'PARAIPABA','TRAIRI', 'SAO LUIS DO CURU']

def transform_municipio(X: pd.DataFrame):
    X['municipio'] = np.select(
        [X['municipio'] == 'FORTALEZA', X['municipio'].isin(RMF)],
        ['Fortaleza', 'RMF'],
        'Interior',
    )
    return X

# pipeline já que são dois passos em sequência
REGIAO_PIPELINE = Pipeline([
    ('regiao', FunctionTransformer(transform_municipio, feature_names_out='one-to-one')),
    ('onehot', OneHotEncoder(categories=[['Fortaleza', 'RMF', 'Interior']],feature_name_combiner=lambda feature, val: f'regiao_{val}')),
])

#### 2.1.9. Transformação dos dados

Vale salientar que essas transformações **não incluem** feature scaling ainda.

In [20]:
from sklearn.compose import ColumnTransformer

LEADS_PREPROCESSOR = ColumnTransformer(
    transformers=[
        # nome da etapa                    transformador usado            # colunas afetadas
        ('drop',                           'drop',                        TO_DROP),
        ('extract_tem_ifood',              IFOOD_FEATURE_EXTRACTOR,       ['ifood_contagem_avaliacoes', 'ifood_faixa_preco']),
        ('impute_google_maps_tem_website', GOOGLE_MAPS_IMPUTER,           ['google_maps_tem_website']),
        ('encode_capital_social',          CAPITAL_SOCIAL_ENCODER,        ['capital_social']),
        ('encode_idade_cnpj',              IDADE_CNPJ_ENCODER,            ['idade_cnpj']),
        ('transform_contagens_serasa',     LOG_TRANSFORMER,               ['serasa_contagem_negativacoes', 'serasa_contagem_protestos']),
        ('transform_natureza_juridica',    NATUREZA_JURIDICA_TRANSFORMER, ['natureza_juridica']),
        ('extract_qtd_credores',           QTD_CREDORES_EXTRACTOR,        ['serasa_credores']),
        ('encode_fonte_cliente',           FONTE_CLIENTE_ENCODER,         ['fonte_cliente']),
        ('encode_regiao',                  REGIAO_PIPELINE,               ['municipio']),
    ],
    verbose_feature_names_out=False,
)

LEADS_PREPROCESSOR

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('drop', ...), ('extract_tem_ifood', ...), ...]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features 

### 2.2. Conjunto comportamental 

#### Uma linha por cliente **por janela**

O alvo (`churned`) é **um por cliente**. Pra ter features de comportamento, resumimos os pedidos de cada cliente.

Pra não misturar passado com futuro, o resumo é feito **por janela de tempo**: as features de uma linha usam **só os pedidos daquela janela**. O mesmo cliente pode aparecer em janelas diferentes, com padrão de compra diferente em cada uma.

As janelas têm o mesmo tamanho de 18 dias e são contadas de trás pra frente a partir da data de corte:

```
janela 0     janela 1     janela 2     janela 3  |  janela 4 
 18 dias      18 dias      18 dias      18 dias  |  a partir de date_threshold
```

In [21]:
pedidos_train_val = COMPORTAMENTAL.train_val.X.copy()
pedidos_test = COMPORTAMENTAL.test.X.copy()

pedidos_train_val['churned'] = COMPORTAMENTAL.train_val.y
pedidos_test['churned'] = COMPORTAMENTAL.test.y

N_JANELAS_TREINO = 4
ULTIMA_JANELA = N_JANELAS_TREINO    # janela do teste

# bordas das janelas de treino+validação: [corte - 72 dias, corte - 54, corte - 36, corte - 18, corte]
bordas = [date_threshold - TEST_TIMEFRAME * k for k in range(N_JANELAS_TREINO, -1, -1)]

# número da janela de cada pedido (0 a 3); -1 = antes da primeira janela (fica de fora)
pedidos_train_val['janela'] = np.searchsorted(bordas, pedidos_train_val['data_entrega'], side='right') - 1
pedidos_test['janela'] = ULTIMA_JANELA


pedidos_train_val = pedidos_train_val[pedidos_train_val['janela'] >= 0]

### Features dos pedidos

Por (cliente, janela):

| feature | descrição |
|---|---|
| `n_pedidos` | quantidade de pedidos na janela |
| `intervalo_medio_dias` | frequência de compra: média de dias entre dias com pedido. `NaN` quando só há 1 dia com pedido |
| `valor_mais_frequente`, `produtos_mais_frequente` | faixa mais frequente do cliente na janela (empate: a faixa maior), como código ordinal 0, 1, 2, ... |

Os pedidos com `valor`/`produtos` nulos (3 no total) são ignorados na contagem da faixa mais frequente.

**Obs:** não foram incluídas a variação da frequência de compra nem a variação do valor ao longo do tempo. Nas janelas de 18 dias, a maioria das linhas tem um único pedido, então não tem como medir variação dentro de uma janela; seria necessário comparar cada janela com a anterior.

In [22]:
ORDEM_VALOR = get_ordered_ranges(pedidos_train_val['valor'])
ORDEM_PRODUTOS = get_ordered_ranges(pedidos_train_val['produtos'])
print('valor:', ORDEM_VALOR)
print('produtos:', ORDEM_PRODUTOS)


def mais_frequente(codigos):
    # código da faixa que mais aparece; empate -> a faixa maior
    contagem = codigos.dropna().value_counts()
    if contagem.empty:
        return np.nan
    return contagem[contagem == contagem.max()].index.max()


def intervalo_medio(datas):
    dias = datas.drop_duplicates().sort_values()     # só dias distintos, em ordem
    return dias.diff().dt.days.mean()                # NaN se só há 1 dia


def resume_pedidos(pedidos):
    # uma linha por (cliente, janela), com o cadastro, as features de pedido e o churned do cliente
    pedidos = pedidos.assign(
        valor_ord=pedidos['valor'].map({faixa: i for i, faixa in enumerate(ORDEM_VALOR)}),
        produtos_ord=pedidos['produtos'].map({faixa: i for i, faixa in enumerate(ORDEM_PRODUTOS)}),
    )

    grupos = pedidos.groupby(['id_cliente', 'janela'])
    linhas = grupos.agg(
        n_pedidos=('id_pedido', 'count'),
        churned=('churned', 'first'),     # é o mesmo em todos os pedidos do cliente
    )
    linhas['intervalo_medio_dias'] = grupos['data_entrega'].apply(intervalo_medio)
    linhas['valor_mais_frequente'] = grupos['valor_ord'].apply(mais_frequente)
    linhas['produtos_mais_frequente'] = grupos['produtos_ord'].apply(mais_frequente)
    linhas = linhas.reset_index()

    # o cadastro é igual em todos os pedidos do cliente - pega de uma linha só
    cadastro = prepara_cadastro(pedidos.drop_duplicates('id_cliente').set_index('id_cliente'))
    return linhas.join(cadastro, on='id_cliente')

valor: ['(0, 500]', '(500, 1000]', '(1000, 2000]', '(2000, 10000000]']
produtos: ['(0, 5]', '(5, 10]', '(10, 10000000]']


In [23]:
from sklearn.compose import make_column_selector

COMPORTAMENTAL_PREPROCESSOR = ColumnTransformer(
    transformers=[
        # nome da etapa                    transformador usado            # colunas afetadas
        ('preprocess_leads',               LEADS_PREPROCESSOR,            make_column_selector(pattern='.*')),
        
    ],
    verbose_feature_names_out=False,
)

COMPORTAMENTAL_PREPROCESSOR

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('preprocess_leads', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The 